# Gold Engine

Functions shared by `Dimensions`, `Facts`, `99_Gold_Run_All` and `90_Gold_Validation`. This notebook only defines
functions: running it (or `%run`-ing it) reads and writes nothing. It expects `00_Gold_Registry` to have been
`%run` first.

## Safety model
- `dry_run=true` (default) prints the ordered plan, the readiness axes, metadata-only source checks (table and
  column existence, types) and, optionally, the SQL. It writes nothing.
- A real run needs an explicit table list; every selected table needs `approved_to_publish=True` and
  `silver_sources_confirmed=True` in `00_Gold_Registry`, and no Silver source may be `NEEDS_PROFILING` or `DEFERRED`.
  There is no override parameter.
- Tables scanning large inputs (`scan="large"`) are **blocked from real runs** (`CANDIDATE_VALIDATION_APPROVED` is False in
  `00_Gold_Registry`): they are not staged, so their results could only be validated after existing Gold was replaced. A reviewed
  candidate-table approach is required first (`docs/gold_build_cost.md`). `confirm_large_scan=true` is still mandatory for their
  read-only validation and would remain mandatory afterwards.
- Publication is `CREATE OR REPLACE TABLE` (default) or, if the schema is unchanged, an in-place overwrite
  (`publish_mode=insert_overwrite`) that keeps the existing table object. Nothing is dropped.
- ANSI mode is switched on for the session so an invalid cast (for example in a join-only expression) fails instead of
  producing NULL. Gold does not narrow identifier columns: they keep the reviewed Silver types.
- Large tables are never staged. Their checks run after publication and each OBSERVE check costs a further pass, so OBSERVE
  checks on large tables run only with `run_observe_checks=true`. Each large table's pass count is printed by the dry run
  (`cost_note`) and listed in `docs/gold_build_cost.md`; they are estimates read from the SQL, not measured cost.
- The session time zone is set explicitly (`session_time_zone`, default `UTC`): date keys come from
  `date_format(timestamp)`, so the time zone decides the calendar day. Assumption to verify: Bronze timestamps
  were read under UTC.

In [0]:
import json
import re

from pyspark.sql import functions as F

PUBLISH_MODES = ("create_or_replace", "insert_overwrite")


def declare_widgets(dbutils, default_tables="all"):
    dbutils.widgets.text("source_schema", DEFAULT_SOURCE_SCHEMA)
    dbutils.widgets.text("target_schema", DEFAULT_TARGET_SCHEMA)
    dbutils.widgets.text("tables", default_tables)
    dbutils.widgets.text("dry_run", "true")
    dbutils.widgets.text("publish_mode", "create_or_replace")
    dbutils.widgets.text("date_start", DEFAULT_DATE_START)
    dbutils.widgets.text("date_end", DEFAULT_DATE_END)
    dbutils.widgets.text("session_time_zone", "UTC")
    dbutils.widgets.text("confirm_large_scan", "false")
    dbutils.widgets.text("run_observe_checks", "false")
    dbutils.widgets.text("stop_on_failure", "true")
    dbutils.widgets.text("show_sql", "true")


def _flag(dbutils, name):
    value = dbutils.widgets.get(name).strip().lower()
    if value not in {"true", "false"}:
        raise ValueError(f"{name} must be true or false")
    return value == "true"


def read_config(dbutils):
    cfg = dict(
        source_schema=dbutils.widgets.get("source_schema").strip(),
        target_schema=dbutils.widgets.get("target_schema").strip(),
        dry_run=_flag(dbutils, "dry_run"),
        publish_mode=dbutils.widgets.get("publish_mode").strip(),
        date_start=dbutils.widgets.get("date_start").strip(),
        date_end=dbutils.widgets.get("date_end").strip(),
        session_time_zone=dbutils.widgets.get("session_time_zone").strip(),
        confirm_large_scan=_flag(dbutils, "confirm_large_scan"),
        run_observe_checks=_flag(dbutils, "run_observe_checks"),
        stop_on_failure=_flag(dbutils, "stop_on_failure"),
        show_sql=_flag(dbutils, "show_sql"),
    )
    for key in ("source_schema", "target_schema"):
        if not re.fullmatch(SCHEMA_PATTERN, cfg[key]):
            raise ValueError(f"{key} must be catalog.schema: {cfg[key]}")
    if cfg["source_schema"] == cfg["target_schema"]:
        raise ValueError("source_schema and target_schema must differ")
    if cfg["publish_mode"] not in PUBLISH_MODES:
        raise ValueError(f"publish_mode must be one of {PUBLISH_MODES}")
    for key in ("date_start", "date_end"):
        if not re.fullmatch(DATE_PATTERN, cfg[key]):
            raise ValueError(f"{key} must be YYYY-MM-DD: {cfg[key]}")
    if cfg["date_start"] >= cfg["date_end"]:
        raise ValueError("date_start must be before date_end")
    if not re.fullmatch(r"[A-Za-z0-9_/+:\-]+", cfg["session_time_zone"]):
        raise ValueError("session_time_zone has unexpected characters")
    raw = dbutils.widgets.get("tables").strip()
    if raw == "all":
        cfg["requested"] = "all"
    elif raw.startswith("["):
        cfg["requested"] = json.loads(raw)
    else:
        cfg["requested"] = [t.strip() for t in raw.split(",") if t.strip()]
    return cfg


def scalar(spark, sql):
    return spark.sql(sql).first()[0]


def column_types(spark, table):
    return {f.name: f.dataType.simpleString() for f in spark.table(table).schema.fields}


def source_problems(spark, S, spec):
    """Metadata-only comparison of the Silver tables with the expected contract (no data is read)."""
    problems = []
    for table, contract in spec["silver_contract"].items():
        name = f"{S}.{table}"
        if not spark.catalog.tableExists(name):
            problems.append(f"missing table {name}")
            continue
        types = column_types(spark, name)
        for column, accepted in contract.items():
            if column not in types:
                problems.append(f"{name}: missing column {column}")
            elif accepted is not None and types[column] not in accepted:
                problems.append(f"{name}.{column}: type {types[column]}, expected {'/'.join(accepted)}")
    return problems


def dependency_problems(spark, G, spec, selected):
    """Gold dimensions read by joins must exist (or be built earlier in this run)."""
    return [f"missing Gold dependency {G}.{d}" for d in spec["gold_deps"]
            if d not in selected and not spark.catalog.tableExists(f"{G}.{d}")]


def apply_session(spark, cfg):
    spark.conf.set("spark.sql.session.timeZone", cfg["session_time_zone"])
    spark.conf.set("spark.sql.ansi.enabled", "true")


def params_of(cfg):
    return {"date_start": cfg["date_start"], "date_end": cfg["date_end"]}


In [0]:
# Validation. Each result: dict(table, check, status, value, detail). status: PASS, FAIL, OBSERVE, SKIPPED, ERROR.

def _result(table, check, status, value=None, detail=""):
    return dict(table=table, check=check, status=status, value=value, detail=detail)


def _run_check(spark, table, check, S, G):
    name, mode, sql_fn = check
    try:
        value = scalar(spark, sql_fn(S, G))
    except Exception as exc:
        return _result(table, name, "ERROR", None, " ".join(str(exc).split())[:200])
    value = 0 if value is None else value
    if mode == "zero":
        return _result(table, name, "PASS" if value == 0 else "FAIL", value)
    return _result(table, name, "OBSERVE", value)


def _dup_groups_sql(target, columns):
    cols = ", ".join(columns)
    return f"SELECT COUNT(*) FROM (SELECT {cols} FROM {target} GROUP BY {cols} HAVING COUNT(*) > 1)"


def dependency_uniqueness(spark, G, table):
    """A dimension joined on its business key must be unique on it, or the join would multiply fact rows."""
    results = []
    for dep in GOLD[table]["gold_deps"]:
        dep_spec = GOLD[dep]
        target = f"{G}.{dep}"
        if not spark.catalog.tableExists(target):
            continue
        dups = scalar(spark, _dup_groups_sql(target, dep_spec["business_key"]))
        results.append(_result(table, f"join_dimension_unique:{dep}", "PASS" if dups == 0 else "FAIL", dups,
                               f"duplicate {dep_spec['business_key']} groups"))
    return results


def validate_table(spark, table, S, G, cfg, confirm_large=False, include_expected=True):
    """Grain, key coverage, join cardinality, row reconciliation and date coverage for one built Gold table."""
    spec = GOLD[table]
    target = f"{G}.{table}"
    params = params_of(cfg)
    large = spec["scan"] == "large"
    out = []
    if not spark.catalog.tableExists(target):
        return [_result(table, "exists", "FAIL", None, f"{target} does not exist")]
    rows = spark.table(target).count()
    out.append(_result(table, "row_count", "OBSERVE", rows))

    if include_expected and (not large or confirm_large):
        expected = scalar(spark, spec["expected_rows"](S, G, params))
        out.append(_result(table, "rows_equal_expected_from_silver", "PASS" if rows == expected else "FAIL", rows,
                           f"expected {expected} (a difference means filtered rows or join multiplication)"))
        if spec.get("source_rows"):
            source = scalar(spark, spec["source_rows"](S, G, params))
            out.append(_result(table, "source_rows_dropped_by_documented_filters", "OBSERVE", source - expected,
                               f"silver rows {source}; expected after filters {expected}"))
    else:
        out.append(_result(table, "rows_equal_expected_from_silver", "SKIPPED", None, "large scan: set confirm_large_scan=true"))

    out.extend(dependency_uniqueness(spark, G, table))

    if large and not confirm_large:
        out.append(_result(table, "grain_and_coverage_profile", "SKIPPED", None, "large scan: set confirm_large_scan=true"))
    else:
        fk_cols = [(fk[0], fk[1], fk[3]) for fk in spec["fks"]]
        selects = ["COUNT(*) AS n"]
        for key in spec["grain_key"]:
            selects.append(f"SUM(CASE WHEN f.{key} IS NULL THEN 1 ELSE 0 END) AS null_grain_{key}")
        for column, _, _ in fk_cols:
            selects.append(f"SUM(CASE WHEN f.{column} IS NULL THEN 1 ELSE 0 END) AS null_fk_{column}")
        joins = []
        for i, dk in enumerate(spec["date_keys"]):
            alias = f"d{i}"
            joins.append(f"LEFT JOIN {G}.dim_date {alias} ON f.{dk} = {alias}.date_key")
            selects.append(f"SUM(CASE WHEN f.{dk} IS NULL THEN 1 ELSE 0 END) AS null_date_{dk}")
            selects.append(f"SUM(CASE WHEN f.{dk} IS NOT NULL AND {alias}.date_key IS NULL THEN 1 ELSE 0 END) AS outside_dim_date_{dk}")
            selects.append(f"MIN(f.{dk}) AS min_{dk}")
            selects.append(f"MAX(f.{dk}) AS max_{dk}")
        sql = f"SELECT {', '.join(selects)} FROM {target} f " + " ".join(joins)
        if not spark.catalog.tableExists(f"{G}.dim_date") and spec["date_keys"]:
            out.append(_result(table, "date_coverage", "FAIL", None, f"{G}.dim_date is needed for date coverage"))
        else:
            profile = spark.sql(sql).first().asDict()
            for key in spec["grain_key"]:
                value = profile[f"null_grain_{key}"] or 0
                out.append(_result(table, f"grain_key_not_null:{key}", "PASS" if value == 0 else "FAIL", value))
            for column, dim, strict in fk_cols:
                value = profile[f"null_fk_{column}"] or 0
                status = ("PASS" if value == 0 else "FAIL") if strict else "OBSERVE"
                out.append(_result(table, f"fk_null:{column}->{dim}", status, value,
                                   "unmatched or missing source value; no 'unknown' member is invented"))
            for dk in spec["date_keys"]:
                outside = profile[f"outside_dim_date_{dk}"] or 0
                out.append(_result(table, f"date_key_outside_dim_date:{dk}", "PASS" if outside == 0 else "FAIL", outside,
                                   f"min {profile[f'min_{dk}']} max {profile[f'max_{dk}']}; NULL keys {profile[f'null_date_{dk}']}"))
        dups = scalar(spark, _dup_groups_sql(target, spec["grain_key"]))
        if spec["key_unique"] == "yes":
            out.append(_result(table, f"grain_unique:{','.join(spec['grain_key'])}", "PASS" if dups == 0 else "FAIL", dups,
                               "duplicate key groups"))
        else:
            out.append(_result(table, f"grain_unique:{','.join(spec['grain_key'])}", "OBSERVE", dups,
                               "uniqueness not profiled: duplicates are reported, never removed"))

    if spec.get("natural_grain_distinct") and (not large or confirm_large):
        natural = scalar(spark, spec["natural_grain_distinct"](S, G, params))
        distinct_keys = scalar(spark, f"SELECT COUNT(DISTINCT {spec['grain_key'][0]}) FROM {target}")
        out.append(_result(table, "key_collision_check:distinct_keys_equal_distinct_natural_records",
                           "PASS" if natural == distinct_keys else "FAIL", distinct_keys,
                           f"distinct natural records in Silver {natural}; a shortfall means a hash collision, a surplus means the key is not a function of the natural record"))

    if spec["group"] == "dimension" and (not large or confirm_large):
        bk_dups = scalar(spark, _dup_groups_sql(target, spec["business_key"]))
        out.append(_result(table, f"business_key_unique:{','.join(spec['business_key'])}",
                           "PASS" if bk_dups == 0 else "FAIL", bk_dups, "join key of the dimension"))

    if not large or confirm_large:
        for check in spec["post_checks"]:
            if check[1] == "observe" and large and not cfg.get("run_observe_checks", False):
                out.append(_result(table, check[0], "SKIPPED", None, "large table: set run_observe_checks=true (one extra pass)"))
                continue
            out.append(_run_check(spark, table, check, S, G))
    return out


def print_results(results):
    for r in results:
        value = "" if r["value"] is None else r["value"]
        print(f"  [{r['status']:<7}] {r['table']:<32} {r['check']:<60} {value!s:<14} {r['detail']}")


def failures(results):
    return [r for r in results if r["status"] in ("FAIL", "ERROR")]


In [0]:
# Publication

def stage_checks(spark, table, S, G, cfg):
    """Small tables only: evaluate the SELECT into a temporary view and validate it BEFORE replacing anything."""
    spec = GOLD[table]
    params = params_of(cfg)
    view = f"_gold_stage_{table}"
    spark.sql(render_select(table, S, G, params)).createOrReplaceTempView(view)
    results = []
    rows = spark.table(view).count()
    expected = scalar(spark, spec["expected_rows"](S, G, params))
    results.append(_result(table, "stage_rows_equal_expected", "PASS" if rows == expected else "FAIL", rows, f"expected {expected}"))
    dups = scalar(spark, _dup_groups_sql(view, spec["grain_key"]))
    results.append(_result(table, "stage_grain_unique", "PASS" if dups == 0 else "FAIL", dups))
    for key in spec["grain_key"]:
        nulls = scalar(spark, f"SELECT COUNT(*) FROM {view} WHERE {key} IS NULL")
        results.append(_result(table, f"stage_not_null:{key}", "PASS" if nulls == 0 else "FAIL", nulls))
    return results


def publish(spark, table, S, G, cfg):
    spec = GOLD[table]
    target = f"{G}.{table}"
    params = params_of(cfg)
    if cfg["publish_mode"] == "insert_overwrite":
        if not spark.catalog.tableExists(target):
            raise RuntimeError(f"insert_overwrite needs an existing {target}; use create_or_replace for the first build")
        frame = spark.sql(render_select(table, S, G, params))
        new_schema = [(f.name, f.dataType.simpleString()) for f in frame.schema.fields]
        old_schema = list(column_types(spark, target).items())
        if new_schema != old_schema:
            raise RuntimeError(f"Schema of {target} would change; use create_or_replace after the governance checks. "
                               f"old={old_schema} new={new_schema}")
        frame.writeTo(target).overwrite(F.lit(True))
    else:
        spark.sql(render_ctas(table, S, G, params))


def run_gold(spark, cfg, group=None):
    """Plan, gate, build and validate. Returns the list of validation results."""
    S, G = cfg["source_schema"], cfg["target_schema"]
    apply_session(spark, cfg)
    plan = build_plan(cfg["requested"], group=group)
    selected = plan["selected"]
    params = params_of(cfg)

    print(f"Source : {S}\nTarget : {G}   (tables are REPLACED when dry_run=false; publish_mode={cfg['publish_mode']})")
    print(f"dry_run={cfg['dry_run']} | session_time_zone={cfg['session_time_zone']} | ansi=true | "
          f"confirm_large_scan={cfg['confirm_large_scan']} | run_observe_checks={cfg['run_observe_checks']}\n")
    problems = {}
    for step in plan["steps"]:
        spec = GOLD[step["table"]]
        axes = (f"code={'Y' if step['code_prepared'] else 'N'} semantics={step['semantics_reviewed']:<7} "
                f"source={step['source_status']:<27} silver_confirmed={'Y' if step['silver_sources_confirmed'] else 'N'} "
                f"validation={step['validation_evidence']:<5} approved={'Y' if step['approved_to_publish'] else 'N'} "
                f"scan={step['scan']}")
        print(f"{step['position']:>2}. {step['table']:<32} {axes}")
        print(f"      silver availability: {step['silver_availability']}")
        if step["cost_note"]:
            print(f"      estimated cost (pass counts read from the SQL, NOT measured): {step['cost_note']}")
        found = source_problems(spark, S, spec) + dependency_problems(spark, G, spec, selected)
        if found:
            problems[step["table"]] = found
    print("\nMetadata checks against the Silver contract (no data read):")
    if problems:
        for table, found in problems.items():
            print(f"  {table}:")
            for line in found:
                print(f"    - {line}")
    else:
        print("  all Silver tables, columns and types match the contract")

    if cfg["show_sql"]:
        for step in plan["steps"]:
            print(f"\n-- {step['table']} ({GOLD[step['table']]['grain']})\n{render_ctas(step['table'], S, G, params)};")

    if cfg["dry_run"]:
        print("\nDRY RUN - nothing was written.")
        try:
            require_executable(plan)
            print("This selection would pass the approval gate.")
        except ValueError as exc:
            print(f"A real run of this selection would be refused: {exc}")
        return []

    require_executable(plan)
    if problems:
        raise RuntimeError(f"Silver contract problems; resolve before building: {problems}")
    large = [s["table"] for s in plan["steps"] if s["scan"] == "large"]
    if large and not cfg["confirm_large_scan"]:
        raise RuntimeError(f"Large-scan tables selected {large}: set confirm_large_scan=true after reviewing cost")

    all_results = []
    for step in plan["steps"]:
        table = step["table"]
        spec = GOLD[table]
        print(f"\n=== {table}")
        gate = dependency_uniqueness(spark, G, table)
        for check in spec["pre_checks"]:
            if check[1] == "observe" and spec["scan"] == "large" and not cfg["run_observe_checks"]:
                gate.append(_result(table, check[0], "SKIPPED", None, "large table: set run_observe_checks=true (one extra pass)"))
            else:
                gate.append(_run_check(spark, table, check, S, G))
        print_results(gate)
        bad = failures(gate)
        if not bad and spec["scan"] == "small":
            staged = stage_checks(spark, table, S, G, cfg)
            print_results(staged)
            bad = failures(staged)
        all_results.extend(gate)
        if bad:
            print(f"BLOCKED {table}: nothing was replaced")
            if cfg["stop_on_failure"]:
                break
            continue
        publish(spark, table, S, G, cfg)
        print(f"published {G}.{table}")
        post = validate_table(spark, table, S, G, cfg, confirm_large=cfg["confirm_large_scan"], include_expected=True)
        print_results(post)
        all_results.extend(post)
        if failures(post) and cfg["stop_on_failure"]:
            print(f"Validation failed after publishing {table}; Delta time travel is the way back (see docs/gold_governance_checks.md)")
            break
    bad = failures(all_results)
    print(f"\nFAILURES: {len(bad)}")
    if bad:
        raise RuntimeError(f"Gold run finished with failures: {[(r['table'], r['check']) for r in bad]}")
    print("Verify governance objects now (91_Gold_Governance_Snapshot) and re-apply tags/masks as documented.")
    return all_results